# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

For Lane 2, our system tackles a hybrid Supervised Binary Classification and Ranking task.  

**Classification:**
 The model operates as a binary classifier to calculate the exact probability ($0.0$ to $1.0$) that a web page belongs to the positive class (at risk of active search traffic decline).

**Ranking:**
Because human capacity is constrained and editors cannot inspect thousands of pages simultaneously, we sort the entire inventory by this calculated probability. This turns a standard classification problem into a highly actionable prioritized review queue.  

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**The Target/Proxy Label:** is_declining_label = (trend_direction == "down").

**Where the Label Comes From:** In our starter slice, this is a rule-derived proxy label based on the current evaluation window. Moving forward into full warehouse iterations, this will mature into a true future observed outcome (e.g., observing a historical 90-day feature window to predict a real traffic drop over the subsequent 30 days). This time-separated separation ensures strict target leakage control

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**The Primary Success Metric:** Precision@K (specifically Precision@50).

**Defense of the Metric:**
Standard accuracy or ROC AUC can look deceptively high while failing operational demands. Because our target business action is a human editorial review, we must optimize for the team's weekly bandwidth ceiling ($K=50$ pages). Precision@50 measures exactly how many of those top 50 flagged items are true decay instances.

**What 'Good' Means:**
Our static baseline rule achieves a Precision@50 of 0.24 (only ~12 correct picks out of 50). A "good" ML system must hit a Precision@50 of 0.70 or higher, reproducing the ~3x performance lift demonstrated by the reference random forest framework

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**The Unit of Analysis (Grain)** is a single pseudonymized content item (one row = one web page). Below, we execute our environment setup script, load the starter dataset slice, isolate the grain, and explicitly sketch out our target vector alongside the core features to inspect its live shape

In [1]:
import os
import sys
import subprocess
import pandas as pd

# 1. Environment Detection and Repository Setup
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Aatka-Saleem/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Navigate up to repository root if launched from notebooks/ or work/notebooks/
    while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")

# 2. Path Validation & Loading
data_path = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(data_path), "Starter CSV not found — are you at the repo root?"

df = pd.read_csv(data_path)

# 3. Create and Sketch the Target Column
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# 4. Display the Frame to Show the Unit of Analysis (One Row = One Content Item)
display_cols = ['content_id', 'client_id', 'impressions_90d', 'content_age_days', 'trend_direction', 'is_declining_label']
print("\n=== SKETCHING THE UNIT OF ANALYSIS DATAFRAME ===")
print(f"DataFrame Shape: {df[display_cols].shape[0]} rows (pages) x {df[display_cols].shape[1]} columns tracked.")
print("\nSample View (First 5 Rows):")
print(df[display_cols].head())


=== SKETCHING THE UNIT OF ANALYSIS DATAFRAME ===
DataFrame Shape: 30000 rows (pages) x 6 columns tracked.

Sample View (First 5 Rows):
             content_id          client_id  impressions_90d  content_age_days  \
0  content_304f48230142  client_f369cb89fc             3803               187   
1  content_a1fb4e703a9e  client_4e07408562            15320               445   
2  content_9aa793d4d895  client_7f2253d7e2            12581               141   
3  content_331d6c4de07b  client_19581e27de            11751               463   
4  content_d99b7a2d90ca  client_3fdba35f04            19140               263   

  trend_direction  is_declining_label  
0            down                   1  
1            down                   1  
2            down                   1  
3          stable                   0  
4            down                   1  


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**A fixed, hand-crafted if-statement rule** (like checking if a page is over 180 days old and dropping in views) breaks under real-world noise. Performance patterns are highly non-linear and context-dependent:

A drop in traffic could indicate search
engine decay, or it could be simple seasonality, a sibling page absorbing traffic via intentional consolidation, or a structural change in Google's SERP layout.

Hard thresholds throw away critical gradient shifts. An arbitrary rule cuts off pages sharply at day 179, whereas a machine learning model evaluates multi-dimensional exposure metrics (impressions_90d, sessions_90d, scroll_rate, position distributions) simultaneously.

ML captures these hidden interactions, providing a fine-grained probability curve that yields a massive precision lift over hard-coded rules.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x]  The notebook runs top to bottom with no errors (Runtime → Run all)
- [x]  No client names, URLs, or private queries anywhere
- [x]  My claims use careful words: observed, measured, directional, decision-support
- [x]  Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.